# ⛏️ Trator de Mineração Massiva: Ebola & Marburg (ChEMBL API)
**Autor:** Antigravity (IA Sênior) & Wesley Capucho
**Objetivo:** Baixar dezenas de milhares de ensaios biológicos (IC50/EC50) direto do banco de dados ChEMBL da Europa, contornando a necessidade de ler PDFs com paywall.
**Feature Engineering:** O script automaticamente puxa a estrutura 2D (SMILES) e já calcula a Matemática (Morgan Fingerprints) para a Fase 2.

In [ ]:
# Instalação das bibliotecas no Colab
!pip install chembl_webresource_client
!pip install rdkit
!pip install pandas tqdm

In [ ]:
# Montando o Google Drive para Checkpoints automáticos
from google.colab import drive
import os

drive.mount('/content/drive')

output_dir = '/content/drive/MyDrive/PROJETO-ANTIVIRAIS-NANOTECNOLOGIA-IA/1-META-ANALISE/1b-DADOS_EXTRAIDOS'
os.makedirs(output_dir, exist_ok=True)
print(f"[*] Dados serão salvos blindados em: {output_dir}")

In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.notebook import tqdm
import time
import json

# 1. Encontrar a L-Polymerase e outros alvos do Ebola/Marburg
print("[*] Conectando ao Banco de Dados Europeu (ChEMBL)...")
target = new_client.target
query = target.search('ebolavirus')
targets = pd.DataFrame.from_dict(query)

print(f"[+] Encontrados {len(targets)} alvos relacionados ao vírus Ebola.")
target_ids = targets['target_chembl_id'].tolist()
print(f"IDs dos Alvos: {target_ids}")

# 2. Garimpar todas as atividades biológicas (Ensaios)
activity = new_client.activity
res = activity.filter(target_chembl_id__in=target_ids).filter(standard_type__in=['IC50', 'EC50', 'Ki', 'Kd'])

print(f"[*] Iniciando extração profunda de {len(res)} ensaios (Isso pode levar horas na madrugada...)")

data = []
checkpoint_file = os.path.join(output_dir, 'dataset_ebola_massive_chembl_CHECKPOINT.csv')

# Contador para não bloquearmos a API
for i, act in enumerate(tqdm(res)):
    # Filtrar apenas os com valor numérico exato e SMILES disponível
    if act['standard_value'] is not None and act['canonical_smiles'] is not None:
        try:
            # Pegando as features brutas
            smiles = act['canonical_smiles']
            valor = float(act['standard_value'])
            tipo = act['standard_type']
            
            # Calculando a Matemática Molecular na hora (Feature Engineering)
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                # Gerar Morgan Fingerprint
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
                fp_bits = list(fp)
                
                # Classificação (Ativo vs Inativo baseada em 10.000 nM = 10 uM)
                is_active = 1 if valor < 10000 else 0
                
                row = {
                    'ChEMBL_ID': act['molecule_chembl_id'],
                    'SMILES': smiles,
                    'Metric': tipo,
                    'Value_nM': valor,
                    'Is_Active': is_active
                }
                
                # Expandindo os bits como colunas para o Pandas
                for bit_idx, bit_val in enumerate(fp_bits):
                    row[f'Bit_{bit_idx}'] = bit_val
                    
                data.append(row)
        except Exception as e:
            pass # Pula moléculas corrompidas do banco

    # Salva a cada 500 moléculas (Checkpoint)
    if (i + 1) % 500 == 0 and len(data) > 0:
        pd.DataFrame(data).to_csv(checkpoint_file, index=False)
        print(f"  [+] Checkpoint de segurança salvo: {i+1} moléculas.")
        time.sleep(1) # Rate limiting de segurança

# 3. Salvamento Final
final_df = pd.DataFrame(data)
final_file = os.path.join(output_dir, 'dataset_ebola_massive_chembl_FINAL.csv')
final_df.to_csv(final_file, index=False)

print("="*60)
print(f"🎉 MINERAÇÃO MASSIVA CONCLUÍDA! {len(final_df)} moléculas extraídas e processadas!")
print(f"Arquivo de Ouro salvo em: {final_file}")
print("="*60)
